In [1]:
# Perform Google Colab installs (if running in Google Colab)
import os

if "COLAB_GPU" in os.environ:
    print("[INFO] Running in Google Colab, installing requirements.")
    !pip install PyMuPDF # for reading PDFs with Python
    !pip install tqdm # for progress bars
    !pip install sentence-transformers # for embedding models
    !pip install accelerate # for quantization model loading
    !pip install bitsandbytes # for quantizing models (less storage space)


[INFO] Running in Google Colab, installing requirements.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00


In [2]:
# Download PDF file
import os
import requests

# Get PDF document
pdf_path = "human-nutrition-text.pdf"

# Download PDF if it doesn't already exist
if not os.path.exists(pdf_path):
  print("File doesn't exist, downloading...")

  # The URL of the PDF you want to download
  url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

  # The local filename to save the downloaded file
  filename = pdf_path

  # Send a GET request to the URL
  response = requests.get(url)

  # Check if the request was successful
  if response.status_code == 200:
      # Open a file in binary write mode and save the content to it
      with open(filename, "wb") as file:
          file.write(response.content)
      print(f"The file has been downloaded and saved as {filename}")
  else:
      print(f"Failed to download the file. Status code: {response.status_code}")
else:
  print(f"File {pdf_path} exists.")

File doesn't exist, downloading...
The file has been downloaded and saved as human-nutrition-text.pdf


In [3]:
# Requires !pip install PyMuPDF, see: https://github.com/pymupdf/pymupdf
import fitz # (pymupdf, found this is better than pypdf for our use case, note: licence is AGPL-3.0, keep that in mind if you want to use any code commercially)
from tqdm.auto import tqdm # for progress bars, requires !pip install tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip() # note: this might be different for each doc (best to experiment)

    # Other potential text formatting functions can go here
    return cleaned_text

# Open PDF and get lines/pages
# Note: this only focuses on text, rather than images/figures etc
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing the page number
        (adjusted), character count, word count, sentence count, token count, and the extracted text
        for each page.
    """
    doc = fitz.open(pdf_path)  # open a document
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # iterate the document pages
        text = page.get_text()  # get plain text encoded as UTF-8
        text = text_formatter(text)
        pages_and_texts.append({"page_number": page_number - 41,  # adjust page numbers since our PDF starts on page 42
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4,  # 1 token = ~4 chars, see: https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them
                                "text": text})
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [4]:
pages_and_texts[50]

{'page_number': 9,
 'page_char_count': 1320,
 'page_word_count': 215,
 'page_sentence_count_raw': 4,
 'page_token_count': 330.0,
 'text': 'Minerals  Major Functions  Macro  Sodium  Fluid balance, nerve transmission, muscle contraction  Chloride  Fluid balance, stomach acid production  Potassium  Fluid balance, nerve transmission, muscle contraction  Calcium  Bone and teeth health maintenance, nerve transmission,  muscle contraction, blood clotting  Phosphorus  Bone and teeth health maintenance, acid-base balance  Magnesium  Protein production, nerve transmission, muscle  contraction  Sulfur  Protein production  Trace  Iron  Carries oxygen, assists in energy production  Zinc  Protein and DNA production, wound healing, growth,  immune system function  Iodine  Thyroid hormone production, growth, metabolism  Selenium  Antioxidant  Copper  Coenzyme, iron metabolism  Manganese  Coenzyme  Fluoride  Bone and teeth health maintenance, tooth decay  prevention  Chromium  Assists insulin in glucos

In [5]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,145,2,199.25,Contents Preface University of Hawai‘i at Mā...


In [6]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 873,
  'page_char_count': 1677,
  'page_word_count': 291,
  'page_sentence_count_raw': 17,
  'page_token_count': 419.25,
  'text': 'children should be provided nutrient-dense food at meal- and  snack-time. However, it is important not to overfeed children, as  this can lead to childhood obesity, which is discussed in the next  section. Parents and other caregivers can turn to the MyPlate  website for guidance: http://www.choosemyplate.gov/.  Macronutrients  For carbohydrates, the Acceptable Macronutrient Distribution  Range (AMDR) is 45–65 percent of daily calories (which is a  recommended daily allowance of 135–195 grams for 1,200 daily  calories). Carbohydrates high in fiber should make up the bulk of  intake. The AMDR for protein is 10–30 percent of daily calories  (30–90 grams for 1,200 daily calories). Children have a high need for  protein to support muscle growth and development. High levels of  essential fatty acids are needed to support growth (although not as

In [7]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,145,2,199.25,Contents Preface University of Hawai‘i at Mā...


In [8]:
# Get stats
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00
std,348.86,560.38,95.76,6.19,140.10
min,-41.00,0.00,1.00,1.00,0.00
25%,260.75,762.00,134.00,4.00,190.50
50%,562.50,1231.50,214.50,10.00,307.88
75%,864.25,1603.50,271.00,14.00,400.88
max,1166.00,2308.00,429.00,32.00,577.00


In [9]:
from spacy.lang.en import English # see https://spacy.io/usage for install instructions

nlp = English()

# Add a sentencizer pipeline, see https://spacy.io/api/sentencizer/
nlp.add_pipe("sentencizer")

# Create a document instance as an example
doc = nlp("This is a sentence. This another sentence.")
assert len(list(doc.sents)) == 2

# Access the sentences of the document
list(doc.sents)

[This is a sentence., This another sentence.]

In [10]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)

    # Make sure all sentences are strings
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    # Count the sentences
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [11]:
# Inspect an example
import random
random.sample(pages_and_texts, k=1)

[{'page_number': 528,
  'page_char_count': 1305,
  'page_word_count': 246,
  'page_sentence_count_raw': 9,
  'page_token_count': 326.25,
  'text': 'The RDA for vitamin A is considered sufficient to support growth  and development, reproduction, vision, and immune system  function while maintaining adequate stores (good for four months)  in the liver.  Table 9.1 Dietary Reference Intakes for Vitamin A  Age Group  RDA Males and Females mcg RAE/ day  UL  Infants (0–6 months)  400*  600  Infants (7–12 months)  500*  600  Children (1–3 years)  300  600  Children (4–8 years)  400  900  Children (9–13 years)  600  1,700  Adolescents (14–18 years)  Males: 900  2,800  Adolescents (14–18 years)  Females: 700  2,800  Adults (> 19 years)  Males: 900  3,000  Adults (> 19 years)  Females: 700  3,000  *denotes Adequate  Intake  Source: Source: Dietary Supplement Fact Sheet: Vitamin A. National  Institutes  of  Health,  Office  of  Dietary  Supplements.  http://ods.od.nih.gov/factsheets/VitaminA-Quick

In [12]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00,10.32
std,348.86,560.38,95.76,6.19,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,4.00,190.50,5.00
50%,562.50,1231.50,214.50,10.00,307.88,10.00
75%,864.25,1603.50,271.00,14.00,400.88,15.00
max,1166.00,2308.00,429.00,32.00,577.00,28.00


In [13]:
# Define split size to turn groups of sentences into chunks
num_sentence_chunk_size = 10

# Create a function that recursively splits a list into desired sizes
def split_list(input_list: list,
               slice_size: int) -> list[list[str]]:
    """
    Splits the input_list into sublists of size slice_size (or as close as possible).

    For example, a list of 17 sentences would be split into two lists of [[10], [7]]
    """
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [14]:
# Sample an example from the group (note: many samples have only 1 chunk as they have <=10 sentences total)
random.sample(pages_and_texts, k=1)

[{'page_number': 959,
  'page_char_count': 1487,
  'page_word_count': 246,
  'page_sentence_count_raw': 12,
  'page_token_count': 371.75,
  'text': 'Sports Nutrition  UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM AND HUMAN NUTRITION PROGRAM  Nutrient Needs for Athletes  Nutrition is essential to your performance during all types of  exercise. The foods consumed in your diet are used to provide  the body with enough energy to fuel an activity regardless of the  intensity of activity. Athletes have different nutritional needs to  support the vigorous level they compete and practice at.  Energy Needs  To determine an athletes nutritional needs, it is important to revisit  the concept of energy metabolism. Energy intake is the foundation  of an athlete’s diet because it supports optimal body functions,  determines the amount of intake of macronutrients and  micronutrients, and assists in the maintaining of body composition.  Energy needs for athletes increase dep

In [15]:
# Create a DataFrame to get stats
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00,10.32,1.53
std,348.86,560.38,95.76,6.19,140.10,6.30,0.64
min,-41.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,260.75,762.00,134.00,4.00,190.50,5.00,1.00
50%,562.50,1231.50,214.50,10.00,307.88,10.00,1.00
75%,864.25,1603.50,271.00,14.00,400.88,15.00,2.00
max,1166.00,2308.00,429.00,32.00,577.00,28.00,3.00


In [16]:
import re

# Split each chunk into its own item
pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]

        # Join the sentences together into a paragraph-like structure, aka a chunk (so they are a single string)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk) # ".A" -> ". A" for any full-stop/capital letter combo
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # Get stats about the chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # 1 token = ~4 characters

        pages_and_chunks.append(chunk_dict)

# How many chunks do we have?
len(pages_and_chunks)

  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [17]:
# View a random sample
random.sample(pages_and_chunks, k=1)

[{'page_number': 1124,
  'sentence_chunk': 'illnesses. Therefore a correct diagnosis involves eliminating other mental illnesses, hormonal imbalances, and nervous system abnormalities. Eliminating these other possibilities involves numerous blood tests, urine tests, and x-rays. Coexisting organ malfunction is also examined. Treatment of any mental illness involves not only the individual, but also family, friends, and a psychiatric counselor. Treating anorexia also involves a dietitian, who helps to provide dietary solutions that often have to be adjusted over time. The goals of treatment for anorexia are to restore a healthy body weight and significantly reduce the behaviors associated with causing the eating disorder. Relapse to an unbalanced diet is high. Many people do recover from anorexia, however most continue to have a lower-than-normal body weight for the rest of their lives. Bulimia Nervosa Bulimia nervosa, like anorexia, is a psychiatric illness that can have severe health c

In [18]:
# Get stats about our chunks
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.00,1843.00,1843.00,1843.00
mean,583.38,734.44,112.33,183.61
std,347.79,447.54,71.22,111.89
min,-41.00,12.00,3.00,3.00
25%,280.50,315.00,44.00,78.75
50%,586.00,746.00,114.00,186.50
75%,890.00,1118.50,173.00,279.62
max,1166.00,1831.00,297.00,457.75


In [19]:
# Show random chunks with under 30 tokens in length
min_token_length = 30
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

Chunk token count: 12.0 | Text: PART V CHAPTER 5. LIPIDS Chapter 5. Lipids | 289
Chunk token count: 17.25 | Text: Updated March 2010. Accessed November 22, 2017. MyPlate Planner | 757
Chunk token count: 20.25 | Text: Honor your health – gentle nutrition       Calories In Versus Calories Out | 1075
Chunk token count: 8.25 | Text: Regulation of Water Balance | 165
Chunk token count: 8.0 | Text: For example, 856 | Toddler Years


In [20]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': -39,
  'sentence_chunk': 'Human Nutrition: 2020 Edition UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN TITCHENAL, SKYLAR HARA, NOEMI ARCEO CAACBAY, WILLIAM MEINKE-LAU, YA-YUN YANG, MARIE KAINOA FIALKOWSKI REVILLA, JENNIFER DRAPER, GEMADY LANGFELDER, CHERYL GIBBY, CHYNA NICOLE CHUN, AND ALLISON CALABRESE',
  'chunk_char_count': 308,
  'chunk_word_count': 42,
  'chunk_token_count': 77.0},
 {'page_number': -38,
  'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program is licensed under a Creative Commons Attribution 4.0 International License, except where otherwise noted.',
  'chunk_char_count': 210,
  'chunk_word_count': 30,
  'chunk_token_count': 52.5}]

In [21]:
# Requires !pip install sentence-transformers
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device="cpu") # choose the device to load the model to (note: GPU will often be *much* faster than CPU)

# Create a list of sentences to turn into numbers
sentences = [
    "The Sentences Transformers library provides an easy and open-source way to create embeddings.",
    "Sentences can be embedded one by one or as a list of strings.",
    "Embeddings are one of the most powerful concepts in machine learning!",
    "Learn to use embeddings well and you'll be well on your way to being an AI engineer."
]

# Sentences are encoded/embedded by calling model.encode()
embeddings = embedding_model.encode(sentences)
embeddings_dict = dict(zip(sentences, embeddings))

# See the embeddings
for sentence, embedding in embeddings_dict.items():
    print("Sentence:", sentence)
    print("Embedding:", embedding)
    print("")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence: The Sentences Transformers library provides an easy and open-source way to create embeddings.
Embedding: [-2.07981542e-02  3.03164609e-02 -2.01218165e-02  6.86483681e-02
 -2.55255792e-02 -8.47690925e-03 -2.07117584e-04 -6.32377267e-02
  2.81605609e-02 -3.33353430e-02  3.02634723e-02  5.30720614e-02
 -5.03525995e-02  2.62288358e-02  3.33313979e-02 -4.51578349e-02
  3.63043919e-02 -1.37115584e-03 -1.20171262e-02  1.14946729e-02
  5.04510403e-02  4.70857136e-02  2.11913101e-02  5.14608137e-02
 -2.03745980e-02 -3.58889401e-02 -6.67836168e-04 -2.94393059e-02
  4.95859571e-02 -1.05639361e-02 -1.52013116e-02 -1.31758384e-03
  4.48196977e-02  1.56023176e-02  8.60379885e-07 -1.21389981e-03
 -2.37978436e-02 -9.09408263e-04  7.34487409e-03 -2.53930292e-03
  5.23370281e-02 -4.68043536e-02  1.66214462e-02  4.71578985e-02
 -4.15599458e-02  9.01910651e-04  3.60278636e-02  3.42215076e-02
  9.68227610e-02  5.94828576e-02 -1.64984539e-02 -3.51250023e-02
  5.92511566e-03 -7.07989442e-04 -2.4103

In [22]:
single_sentence = "Yo! How cool are embeddings?"
single_embedding = embedding_model.encode(single_sentence)
print(f"Sentence: {single_sentence}")
print(f"Embedding:\n{single_embedding}")
print(f"Embedding size: {single_embedding.shape}")

Sentence: Yo! How cool are embeddings?
Embedding:
[-1.97447781e-02 -4.51084413e-03 -4.98483004e-03  6.55444413e-02
 -9.87675134e-03  2.72835735e-02  3.66425961e-02 -3.30222002e-03
  8.50077812e-03  8.24948121e-03 -2.28498094e-02  4.02430333e-02
 -5.75200431e-02  6.33693039e-02  4.43207212e-02 -4.49507348e-02
  1.25284083e-02 -2.52012648e-02 -3.55292447e-02  1.29559273e-02
  8.67022667e-03 -1.92916617e-02  3.55631020e-03  1.89505704e-02
 -1.47127742e-02 -9.39850230e-03  7.64169870e-03  9.62192286e-03
 -5.98929403e-03 -3.90169099e-02 -5.47824427e-02 -5.67459874e-03
  1.11645572e-02  4.08067740e-02  1.76319088e-06  9.15294047e-03
 -8.77265353e-03  2.39382703e-02 -2.32784711e-02  8.04999098e-02
  3.19176838e-02  5.12596173e-03 -1.47708440e-02 -1.62524674e-02
 -6.03212789e-02 -4.35689725e-02  4.51212153e-02 -1.79053731e-02
  2.63366923e-02 -3.47866789e-02 -8.89175199e-03 -5.47675341e-02
 -1.24372207e-02 -2.38606650e-02  8.33496228e-02  5.71243018e-02
  1.13328760e-02 -1.49594788e-02  9.2037

In [23]:
%%time

# Uncomment to see how long it takes to create embeddings on CPU
# # Make sure the model is on the CPU
# embedding_model.to("cpu")

# # Embed each chunk one by one
# for item in tqdm(pages_and_chunks_over_min_token_len):
#     item["embedding"] = embedding_model.encode(item["sentence_chunk"])

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 5.48 µs


In [24]:
%%time

# Send the model to the GPU
embedding_model.to("cuda") # requires a GPU installed, for reference on my local machine, I'm using a NVIDIA RTX 4090

# Create embeddings one by one on the GPU
for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

  0%|          | 0/1680 [00:00<?, ?it/s]

CPU times: user 28.3 s, sys: 392 ms, total: 28.7 s
Wall time: 29.6 s


In [25]:
# Turn text chunks into a single list
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

In [26]:
%%time

# Embed all texts in batches
text_chunk_embeddings = embedding_model.encode(text_chunks,
                                               batch_size=32, # you can use different batch sizes here for speed/performance, I found 32 works well for this use case
                                               convert_to_tensor=True) # optional to return embeddings as tensor instead of array

text_chunk_embeddings

CPU times: user 22.9 s, sys: 51.5 ms, total: 23 s
Wall time: 22.6 s


tensor([[ 0.0674,  0.0902, -0.0051,  ..., -0.0221, -0.0232,  0.0126],
        [ 0.0552,  0.0592, -0.0166,  ..., -0.0120, -0.0103,  0.0227],
        [ 0.0280,  0.0340, -0.0206,  ..., -0.0054,  0.0213,  0.0313],
        ...,
        [ 0.0771,  0.0098, -0.0122,  ..., -0.0409, -0.0752, -0.0241],
        [ 0.1030, -0.0165,  0.0083,  ..., -0.0574, -0.0283, -0.0295],
        [ 0.0864, -0.0125, -0.0113,  ..., -0.0522, -0.0337, -0.0299]],
       device='cuda:0')

In [27]:
# Save embeddings to file
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [28]:
# Import saved file and view
text_chunks_and_embedding_df_load = pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-39,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,[ 6.74242675e-02 9.02281404e-02 -5.09548886e-...
1,-38,Human Nutrition: 2020 Edition by University of...,210,30,52.50,[ 5.52156419e-02 5.92139773e-02 -1.66167244e-...
2,-37,Contents Preface University of Hawai‘i at Māno...,766,114,191.50,[ 2.79801842e-02 3.39813754e-02 -2.06426680e-...
3,-36,Lifestyles and Nutrition University of Hawai‘i...,941,142,235.25,[ 6.82566985e-02 3.81275043e-02 -8.46854225e-...
4,-35,The Cardiovascular System University of Hawai‘...,998,152,249.50,[ 3.30264494e-02 -8.49763490e-03 9.57159605e-...


In [29]:
import random

import torch
import numpy as np
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

# Import texts and embedding df
text_chunks_and_embedding_df = pd.read_csv("text_chunks_and_embeddings_df.csv")

# Convert embedding column back to np.array (it got converted to string when it got saved to CSV)
text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

# Convert texts and embedding df to list of dicts
pages_and_chunks = text_chunks_and_embedding_df.to_dict(orient="records")

# Convert embeddings to torch tensor and send to device (note: NumPy arrays are float64, torch tensors are float32 by default)
embeddings = torch.tensor(np.array(text_chunks_and_embedding_df["embedding"].tolist()), dtype=torch.float32).to(device)
embeddings.shape

torch.Size([1680, 768])

In [30]:
text_chunks_and_embedding_df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-39,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,"[0.0674242675, 0.0902281404, -0.00509548886, -..."
1,-38,Human Nutrition: 2020 Edition by University of...,210,30,52.50,"[0.0552156419, 0.0592139773, -0.0166167244, -0..."
2,-37,Contents Preface University of Hawai‘i at Māno...,766,114,191.50,"[0.0279801842, 0.0339813754, -0.020642668, 0.0..."
3,-36,Lifestyles and Nutrition University of Hawai‘i...,941,142,235.25,"[0.0682566985, 0.0381275043, -0.00846854225, -..."
4,-35,The Cardiovascular System University of Hawai‘...,998,152,249.50,"[0.0330264494, -0.0084976349, 0.00957159605, -..."


In [31]:
embeddings[0]

tensor([ 6.7424e-02,  9.0228e-02, -5.0955e-03, -3.1755e-02,  7.3908e-02,
         3.5198e-02, -1.9799e-02,  4.6769e-02,  5.3573e-02,  5.0123e-03,
         3.3393e-02, -1.6222e-03,  1.7608e-02,  3.6265e-02, -3.1668e-04,
        -1.0712e-02,  1.5426e-02,  2.6218e-02,  2.7765e-03,  3.6494e-02,
        -4.4411e-02,  1.8936e-02,  4.9012e-02,  1.6402e-02, -4.8578e-02,
         3.1829e-03,  2.7299e-02, -2.0476e-03, -1.2283e-02, -7.2805e-02,
         1.2045e-02,  1.0730e-02,  2.1000e-03, -8.1777e-02,  2.6783e-06,
        -1.8143e-02, -1.2080e-02,  2.4717e-02, -6.2747e-02,  7.3544e-02,
         2.2162e-02, -3.2877e-02, -1.8010e-02,  2.2295e-02,  5.6137e-02,
         1.7951e-03,  5.2593e-02, -3.3174e-03, -8.3387e-03, -1.0628e-02,
         2.3192e-03, -2.2393e-02, -1.5301e-02, -9.9306e-03,  4.6532e-02,
         3.5747e-02, -2.5476e-02,  2.6369e-02,  3.7491e-03, -3.8268e-02,
         2.5833e-02,  4.1287e-02,  2.5818e-02,  3.3297e-02, -2.5178e-02,
         4.5152e-02,  4.4907e-04, -9.9662e-02,  4.9

In [32]:
from sentence_transformers import util, SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device=device) # choose the device to load the model to

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [33]:
# 1. Define the query
# Note: This could be anything. But since we're working with a nutrition textbook, we'll stick with nutrition-based queries.
query = "macronutrients functions"
print(f"Query: {query}")

# 2. Embed the query to the same numerical space as the text examples
# Note: It's important to embed your query with the same model you embedded your examples with.
query_embedding = embedding_model.encode(query, convert_to_tensor=True)

# 3. Get similarity scores with the dot product (we'll time this for fun)
from time import perf_counter as timer

start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=embeddings)[0]
end_time = timer()

print(f"Time take to get scores on {len(embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

# 4. Get the top-k results (we'll keep this to 5)
top_results_dot_product = torch.topk(dot_scores, k=5)
top_results_dot_product

Query: macronutrients functions
Time take to get scores on 1680 embeddings: 0.02183 seconds.


torch.return_types.topk(
values=tensor([0.6926, 0.6738, 0.6646, 0.6536, 0.6473], device='cuda:0'),
indices=tensor([42, 47, 41, 51, 46], device='cuda:0'))

In [34]:
larger_embeddings = torch.randn(100*embeddings.shape[0], 768).to(device)
print(f"Embeddings shape: {larger_embeddings.shape}")

# Perform dot product across 168,000 embeddings
start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=larger_embeddings)[0]
end_time = timer()

print(f"Time take to get scores on {len(larger_embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

Embeddings shape: torch.Size([168000, 768])
Time take to get scores on 168000 embeddings: 0.00050 seconds.


In [35]:
# Define helper function to print wrapped text
import textwrap

def print_wrapped(text, wrap_length=80):
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)

In [36]:
print(f"Query: '{query}'\n")
print("Results:")
# Loop through zipped together scores and indicies from torch.topk
for score, idx in zip(top_results_dot_product[0], top_results_dot_product[1]):
    print(f"Score: {score:.4f}")
    # Print relevant sentence chunk (since the scores are in descending order, the most relevant chunk will be first)
    print("Text:")
    print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
    # Print the page number too so we can reference the textbook further (and check the results)
    print(f"Page number: {pages_and_chunks[idx]['page_number']}")
    print("\n")

Query: 'macronutrients functions'

Results:
Score: 0.6926
Text:
Macronutrients Nutrients that are needed in large amounts are called
macronutrients. There are three classes of macronutrients: carbohydrates,
lipids, and proteins. These can be metabolically processed into cellular energy.
The energy from macronutrients comes from their chemical bonds. This chemical
energy is converted into cellular energy that is then utilized to perform work,
allowing our bodies to conduct their basic functions. A unit of measurement of
food energy is the calorie. On nutrition food labels the amount given for
“calories” is actually equivalent to each calorie multiplied by one thousand. A
kilocalorie (one thousand calories, denoted with a small “c”) is synonymous with
the “Calorie” (with a capital “C”) on nutrition food labels. Water is also a
macronutrient in the sense that you require a large amount of it, but unlike the
other macronutrients, it does not yield calories. Carbohydrates Carbohydrates
are 

In [37]:
def retrieve_relevant_resources(query, embeddings, model=embedding_model, n_resources_to_return=5):
    """
    Takes a query, embeds it, compares it with document embeddings,
    and returns the top-k most relevant chunks.
    """
    query_embedding = model.encode(query, convert_to_tensor=True)
    query_embedding = query_embedding.to(embeddings.device)

    dot_scores = util.dot_score(query_embedding, embeddings)[0]

    scores, indices = torch.topk(dot_scores, k=n_resources_to_return)

    return scores, indices


def print_top_results_and_scores(query, embeddings, pages_and_chunks, n_resources_to_return=5):
    """
    Prints top-k retrieved chunks with their similarity scores.
    """
    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=embeddings,
        n_resources_to_return=n_resources_to_return
    )

    print(f"Query: {query}\n")
    print("Top retrieved chunks:\n")

    for score, idx in zip(scores, indices):
        idx = idx.item()
        print(f"Score: {score:.4f}")
        print("Text:")
        print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
        print(f"Page number: {pages_and_chunks[idx]['page_number']}")
        print("-" * 80)

In [38]:
query = "symptoms of pellagra"

print_top_results_and_scores(
    query=query,
    embeddings=embeddings,
    pages_and_chunks=pages_and_chunks,
    n_resources_to_return=5
)

Query: symptoms of pellagra

Top retrieved chunks:

Score: 0.5000
Text:
Niacin deficiency is commonly known as pellagra and the symptoms include
fatigue, decreased appetite, and indigestion.  These symptoms are then commonly
followed by the four D’s: diarrhea, dermatitis, dementia, and sometimes death.
Figure 9.12  Conversion of Tryptophan to Niacin Water-Soluble Vitamins | 565
Page number: 565
--------------------------------------------------------------------------------
Score: 0.3741
Text:
car. Does it drive faster with a half-tank of gas or a full one?It does not
matter; the car drives just as fast as long as it has gas. Similarly, depletion
of B vitamins will cause problems in energy metabolism, but having more than is
required to run metabolism does not speed it up. Buyers of B-vitamin supplements
beware; B vitamins are not stored in the body and all excess will be flushed
down the toilet along with the extra money spent. B vitamins are naturally
present in numerous foods, and m

In [39]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM loaded successfully.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully.


In [40]:
def generate_llm_answer(input_text, max_new_tokens=256):
    messages = [
        {"role": "user", "content": input_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_device = next(llm_model.parameters()).device

    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Only decode newly generated tokens, not the original prompt
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer


test_question = "What are macronutrients?"
answer = generate_llm_answer(test_question)

print("Question:", test_question)
print("Answer:")
print(answer)

Question: What are macronutrients?
Answer:
Macronutrients are nutrients that the human body needs in large amounts to function properly and maintain health. They include carbohydrates, proteins, fats, vitamins, and minerals.

1. Carbohydrates: These provide energy for the body's cells and tissues. They can be broken down into glucose (sugar) which is used as fuel for the brain and muscles.

2. Proteins: These are essential for building and repairing tissues, enzymes, hormones, and antibodies. They also help transport oxygen throughout the body.

3. Fats: These are important for insulation, hormone production, and cell membrane structure. They also play a role in nutrient absorption and storage.

4. Vitamins: These are organic compounds that act as coenzymes or cofactors in metabolic reactions. They help regulate various bodily functions such as growth, development, and immune response.

5. Minerals: These are inorganic elements that are crucial for maintaining proper bodily functions. 

In [41]:
def prompt_formatter(query, context_items):
    """
    Formats retrieved PDF chunks into a prompt for the LLM.
    """

    context = "\n\n".join(
        [
            f"Context {i+1} | Page {item['page_number']} | Score {item['score']:.4f}\n"
            f"{item['sentence_chunk']}"
            for i, item in enumerate(context_items)
        ]
    )

    prompt = f"""
You are a helpful assistant answering questions based on a PDF document.

Use only the context below to answer the question.
Do not use outside knowledge.
If the answer is not available in the context, say:
"The document does not provide enough information."

Context:
{context}

Question:
{query}

Answer:
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

In [42]:
def ask(query, n_resources_to_return=2, max_new_tokens=256):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks from PDF
    2. Add retrieved chunks to the prompt as context
    3. Generate an answer using the LLM
    """

    # 1. Retrieve relevant chunks
    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=embeddings,
        n_resources_to_return=n_resources_to_return
    )

    # 2. Prepare context items
    context_items = []

    for score, idx in zip(scores, indices):
        idx = idx.item()
        item = pages_and_chunks[idx].copy()
        item["score"] = score.item()
        context_items.append(item)

    # 3. Format prompt with retrieved context
    prompt = prompt_formatter(
        query=query,
        context_items=context_items
    )

    # 4. Tokenize prompt
    model_device = next(llm_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    # 5. Generate answer
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # 6. Decode only newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer, context_items

In [43]:
query = "What are the symptoms of pellagra?"

answer, context_items = ask(
    query=query,
    n_resources_to_return=2,
    max_new_tokens=256
)

print("Query:")
print(query)

print("\nRAG Answer:")
print(answer)

print("\nRetrieved Contexts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print_wrapped(item["sentence_chunk"])
    print("-" * 80)

Query:
What are the symptoms of pellagra?

RAG Answer:
The symptoms of pellagra include fatigue, decreased appetite, indigestion, diarrhea, dermatitis (skin issues), dementia, and sometimes death.

Retrieved Contexts:
Score: 0.4655
Page number: 565
Niacin deficiency is commonly known as pellagra and the symptoms include
fatigue, decreased appetite, and indigestion.  These symptoms are then commonly
followed by the four D’s: diarrhea, dermatitis, dementia, and sometimes death.
Figure 9.12  Conversion of Tryptophan to Niacin Water-Soluble Vitamins | 565
--------------------------------------------------------------------------------
Score: 0.3476
Page number: 591
car. Does it drive faster with a half-tank of gas or a full one?It does not
matter; the car drives just as fast as long as it has gas. Similarly, depletion
of B vitamins will cause problems in energy metabolism, but having more than is
required to run metabolism does not speed it up. Buyers of B-vitamin supplements
beware; B vit

In [44]:
query = "What are macronutrients, and what roles do they play in the human body?"

answer, context_items = ask(
    query=query,
    n_resources_to_return=2,
    max_new_tokens=256
)

print("Query:")
print(query)

print("\nRAG Answer:")
print(answer)

print("\nRetrieved Contexts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print_wrapped(item["sentence_chunk"])
    print("-" * 80)

Query:
What are macronutrients, and what roles do they play in the human body?

RAG Answer:
Macronutrients are nutrients that are needed in large amounts and include carbohydrates, lipids, and proteins. They can be metabolically processed into cellular energy and come from the chemical bonds within these macronutrients. The energy from macronutrients is converted into cellular energy that is used to perform various bodily functions. Macronutrients are crucial for providing energy, aiding in metabolism, and supporting overall health.

Retrieved Contexts:
Score: 0.7387
Page number: 5
Macronutrients Nutrients that are needed in large amounts are called
macronutrients. There are three classes of macronutrients: carbohydrates,
lipids, and proteins. These can be metabolically processed into cellular energy.
The energy from macronutrients comes from their chemical bonds. This chemical
energy is converted into cellular energy that is then utilized to perform work,
allowing our bodies to conduc

In [45]:
import pandas as pd

evaluation_queries = [
    "What are the symptoms of pellagra?",
    "What are macronutrients, and what roles do they play in the human body?",
    "How does water help the human body?",
    "What is the difference between macronutrients and micronutrients?",
    "What are water-soluble vitamins?"
]

rag_results = []

for query in evaluation_queries:
    answer, context_items = ask(
        query=query,
        n_resources_to_return=2,
        max_new_tokens=256
    )

    retrieved_pages = [item["page_number"] for item in context_items]
    retrieval_scores = [round(item["score"], 4) for item in context_items]

    rag_results.append({
        "Query": query,
        "RAG Answer": answer,
        "Retrieved Pages": retrieved_pages,
        "Retrieval Scores": retrieval_scores
    })

rag_results_df = pd.DataFrame(rag_results)
rag_results_df

,Query,RAG Answer,Retrieved Pages,Retrieval Scores
0,What are the symptoms of pellagra?,"The symptoms of pellagra include fatigue, decr...","[565, 591]","[0.4655, 0.3476]"
1,"What are macronutrients, and what roles do the...",Macronutrients are nutrients that are needed i...,"[5, 8]","[0.7387, 0.7118]"
2,How does water help the human body?,Water helps the human body through several key...,"[156, 150]","[0.6914, 0.6348]"
3,What is the difference between macronutrients ...,The main difference between macronutrients and...,"[5, 8]","[0.6544, 0.5499]"
4,What are water-soluble vitamins?,The document does not provide enough information.,"[592, 550]","[0.7362, 0.7234]"


In [46]:
nutrition_dataset = {
    "name": "Human Nutrition PDF",
    "pdf_path": "human-nutrition-text.pdf",
    "chunks": pages_and_chunks,
    "embeddings": embeddings
}

print(nutrition_dataset["name"])
print("Number of chunks:", len(nutrition_dataset["chunks"]))
print("Embedding shape:", nutrition_dataset["embeddings"].shape)

Human Nutrition PDF
Number of chunks: 1680
Embedding shape: torch.Size([1680, 768])


In [47]:
import os
import requests

poetry_pdf_path = "leaves_of_grass.pdf"
poetry_url = "https://www.waltwhitman.com/leaves-of-grass.pdf"

if not os.path.exists(poetry_pdf_path):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(poetry_url, headers=headers)

    if response.status_code == 200:
        with open(poetry_pdf_path, "wb") as f:
            f.write(response.content)
        print("Poetry PDF downloaded successfully.")
    else:
        print("Failed to download PDF.")
        print("Status code:", response.status_code)
else:
    print("Poetry PDF already exists.")

Poetry PDF downloaded successfully.


In [48]:
import fitz

doc = fitz.open(poetry_pdf_path)

print("Number of pages:", len(doc))

sample_page = doc.load_page(5)
sample_text = sample_page.get_text("text")

doc.close()

print(sample_text[:1500])

Number of pages: 473
 
Free eBooks at www.WaltWhitman.com 
4 
BOOK XXI. DRUM-TAPS .......................................................................................................................221 
FIRST O SONGS FOR A PRELUDE .....................................................................................................221 
EIGHTEEN SIXTY-ONE .........................................................................................................................223 
BEAT! BEAT! DRUMS! ...........................................................................................................................223 
FROM PAUMANOK STARTING I FLY LIKE A BIRD .....................................................................223 
SONG OF THE BANNER AT DAYBREAK ..........................................................................................225 
RISE O DAYS FROM YOUR FATHOMLESS DEEPS ........................................................................228 
VIRGINIA—TH

In [49]:
import fitz

doc = fitz.open(poetry_pdf_path)

for page_num in [5, 10, 15, 20, 25, 30, 35, 40, 50]:
    page = doc.load_page(page_num)
    text = page.get_text("text")
    print("=" * 100)
    print("PAGE:", page_num)
    print(text[:1000])

doc.close()

PAGE: 5
 
Free eBooks at www.WaltWhitman.com 
4 
BOOK XXI. DRUM-TAPS .......................................................................................................................221 
FIRST O SONGS FOR A PRELUDE .....................................................................................................221 
EIGHTEEN SIXTY-ONE .........................................................................................................................223 
BEAT! BEAT! DRUMS! ...........................................................................................................................223 
FROM PAUMANOK STARTING I FLY LIKE A BIRD .....................................................................223 
SONG OF THE BANNER AT DAYBREAK ..........................................................................................225 
RISE O DAYS FROM YOUR FATHOMLESS DEEPS ........................................................................228 
VIRGINIA—THE WEST ......

In [50]:
import fitz

search_terms = [
    "ONE'S-SELF I SING",
    "I celebrate myself",
    "Song of Myself",
    "BOOK I. INSCRIPTIONS"
]

doc = fitz.open(poetry_pdf_path)

for page_num in range(len(doc)):
    text = doc.load_page(page_num).get_text("text")
    lower_text = text.lower()

    if any(term.lower() in lower_text for term in search_terms):
        print("=" * 100)
        print("Possible poetry start page:", page_num)
        print(text[:1200])
        break

doc.close()

Possible poetry start page: 1
 
 
Contents 
LEAVES OF GRASS ..................................................................................................................................11 
BOOK I. INSCRIPTIONS ..........................................................................................................................11 
ONE’S-SELF I SING ..................................................................................................................................11 
AS I PONDER’D IN SILENCE .................................................................................................................11 
IN CABIN’D SHIPS AT SEA .....................................................................................................................12 
TO FOREIGN LANDS ...............................................................................................................................12 
TO A HISTORIAN..........................................................

In [51]:
import fitz
import re
import pandas as pd
from tqdm.auto import tqdm

def clean_poetry_text(text: str) -> str:
    """
    Cleans poetry text while preserving line breaks.
    """
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def should_skip_poetry_line(line: str) -> bool:
    """
    Removes table-of-contents lines, page numbers, URLs, and decorative lines.
    """
    line = line.strip()

    if line == "":
        return True

    if re.fullmatch(r"\d+", line):
        return True

    if len(line) <= 2:
        return True

    if "Free eBooks" in line or "WaltWhitman.com" in line:
        return True

    # Table of contents style lines: TITLE .......... 221
    if re.search(r"\.{5,}\s*\d+$", line):
        return True

    return False


def open_and_read_poetry_pdf(pdf_path: str, start_page: int = 0, end_page: int | None = None) -> list[dict]:
    """
    Reads a poetry PDF while preserving line structure.
    """
    doc = fitz.open(pdf_path)

    if end_page is None:
        end_page = len(doc)

    pages_and_texts = []

    for page_number in tqdm(range(start_page, end_page)):
        page = doc.load_page(page_number)
        text = page.get_text("text")
        text = clean_poetry_text(text)

        pages_and_texts.append({
            "page_number": page_number,
            "page_char_count": len(text),
            "page_word_count": len(text.split()),
            "page_token_count": len(text) / 4,
            "text": text
        })

    doc.close()
    return pages_and_texts


def build_poetry_rag_dataset_from_pdf(
    pdf_path: str,
    dataset_name: str,
    embedding_model,
    device,
    start_page: int = 0,
    end_page: int | None = None,
    lines_per_chunk: int = 10,
    min_token_length: int = 8
):
    """
    Builds a RAG dataset from a poetry PDF using line-based chunking.
    """

    pages_and_texts = open_and_read_poetry_pdf(
        pdf_path=pdf_path,
        start_page=start_page,
        end_page=end_page
    )

    poetry_chunks = []

    for page in tqdm(pages_and_texts):
        raw_lines = page["text"].split("\n")

        lines = []

        for line in raw_lines:
            line = line.strip()

            if should_skip_poetry_line(line):
                continue

            lines.append(line)

        for i in range(0, len(lines), lines_per_chunk):
            chunk_lines = lines[i:i + lines_per_chunk]
            chunk_text = "\n".join(chunk_lines).strip()

            if len(chunk_text) == 0:
                continue

            chunk_dict = {
                "page_number": page["page_number"],
                "sentence_chunk": chunk_text,
                "chunk_char_count": len(chunk_text),
                "chunk_word_count": len(chunk_text.split()),
                "chunk_token_count": len(chunk_text) / 4,
                "document_type": "poetry"
            }

            poetry_chunks.append(chunk_dict)

    df_poetry_chunks = pd.DataFrame(poetry_chunks)

    poetry_chunks_over_min_token_len = df_poetry_chunks[
        df_poetry_chunks["chunk_token_count"] > min_token_length
    ].to_dict(orient="records")

    text_chunks = [
        item["sentence_chunk"]
        for item in poetry_chunks_over_min_token_len
    ]

    poetry_embeddings = embedding_model.encode(
        text_chunks,
        batch_size=32,
        convert_to_tensor=True
    ).to(device)

    poetry_dataset = {
        "name": dataset_name,
        "pdf_path": pdf_path,
        "chunks": poetry_chunks_over_min_token_len,
        "embeddings": poetry_embeddings,
        "document_type": "poetry"
    }

    print(f"Dataset name: {dataset_name}")
    print(f"Document type: poetry")
    print(f"Start page: {start_page}")
    print(f"Number of chunks: {len(poetry_dataset['chunks'])}")
    print(f"Embedding shape: {poetry_dataset['embeddings'].shape}")

    return poetry_dataset

In [52]:
poetry_dataset = build_poetry_rag_dataset_from_pdf(
    pdf_path=poetry_pdf_path,
    dataset_name="Leaves of Grass Poetry PDF",
    embedding_model=embedding_model,
    device=device,
    start_page=11,
    lines_per_chunk=10,
    min_token_length=8
)

  0%|          | 0/462 [00:00<?, ?it/s]

  0%|          | 0/462 [00:00<?, ?it/s]

Dataset name: Leaves of Grass Poetry PDF
Document type: poetry
Start page: 11
Number of chunks: 1352
Embedding shape: torch.Size([1352, 768])


In [53]:
len(poetry_dataset["chunks"])

1352

In [54]:
import random

for item in random.sample(poetry_dataset["chunks"], k=5):
    print("Page:", item["page_number"])
    print("Token count:", round(item["chunk_token_count"], 2))
    print(item["sentence_chunk"])
    print("-" * 100)

Page: 348
Token count: 161.0
Scattering for good the cloud that hung so long, that weigh’d so long upon the mind of man,
The doubt, suspicion, dread, of gradual, certain decadence of man;
Thee in thy larger, saner brood of female, male—thee in thy athletes, moral, spiritual, South, North, West,
East,
(To thy immortal breasts, Mother of All, thy every daughter, son, endear’d alike, forever equal,)
Thee in thy own musicians, singers, artists, unborn yet, but certain,
Thee in thy moral wealth and civilization, (until which thy proudest material civilization must remain in
vain,)
Thee in thy all-supplying, all-enclosing worship—thee in no single bible, saviour, merely,
----------------------------------------------------------------------------------------------------
Page: 230
Token count: 150.0
What was that swell I saw on the ocean? behold what comes here, How it climbs with daring feet and
hands—how it dashes!
How the true thunder bellows after the lightning—how bright the flashes of l

In [55]:
bad_terms = ["Free eBooks", "WaltWhitman.com", "Contents"]

bad_chunks = [
    item for item in poetry_dataset["chunks"]
    if any(term.lower() in item["sentence_chunk"].lower() for term in bad_terms)
]

print("Number of suspicious chunks:", len(bad_chunks))

for item in bad_chunks[:5]:
    print("Page:", item["page_number"])
    print(item["sentence_chunk"])
    print("-" * 80)

Number of suspicious chunks: 0


In [56]:
poetry_df = pd.DataFrame(poetry_dataset["chunks"])

poetry_df[[
    "chunk_char_count",
    "chunk_word_count",
    "chunk_token_count"
]].describe().round(2)

,chunk_char_count,chunk_word_count,chunk_token_count
count,1352.00,1352.00,1352.00
mean,523.18,92.10,130.79
std,185.84,33.36,46.46
min,43.00,7.00,10.75
25%,442.00,76.00,110.50
50%,573.50,100.00,143.38
75%,652.00,116.00,163.00
max,933.00,160.00,233.25


In [57]:
poetry_query = "Which lines mention death or mortality?"

print_top_results_and_scores(
    query=poetry_query,
    embeddings=poetry_dataset["embeddings"],
    pages_and_chunks=poetry_dataset["chunks"],
    n_resources_to_return=3
)

Query: Which lines mention death or mortality?

Top retrieved chunks:

Score: 0.6522
Text:
The houses full of life were equally full of death, (this house is now,) The
streets, the shipping, the places of amusement, the Chicago, Boston,
Philadelphia, the Mannahatta, were as full of the dead as of the living, And
fuller, O vastly fuller of the dead than of the living; And what I dream’d I
will henceforth tell to every person and age, And I stand henceforth bound to
what I dream’d, And now I am willing to disregard burial-places and dispense
with them, And if the memorials of the dead were put up indifferently
everywhere, even in the room where I eat or sleep, I should be satisfied, And if
the corpse of any one I love, or if my own corpse, be duly render’d to powder
and pour’d in the sea,
Page number: 337
--------------------------------------------------------------------------------
Score: 0.6432
Text:
And I knew death, its thought, and the sacred knowledge of death. Then with the
know

In [58]:
poetry_query = "Which lines express loneliness or sadness?"

print_top_results_and_scores(
    query=poetry_query,
    embeddings=poetry_dataset["embeddings"],
    pages_and_chunks=poetry_dataset["chunks"],
    n_resources_to_return=3
)

Query: Which lines express loneliness or sadness?

Top retrieved chunks:

Score: 0.5901
Text:
Who was not proud of his songs, but of the measureless ocean of love within him,
and freely pour’d it forth, Who often walk’d lonesome walks thinking of his dear
friends, his lovers, Who pensive away from one he lov’d often lay sleepless and
dissatisfied at night, Who knew too well the sick, sick dread lest the one he
lov’d might secretly be indifferent to him,
Page number: 94
--------------------------------------------------------------------------------
Score: 0.5706
Text:
O brown halo in the sky near the moon, drooping upon the sea! O troubled
reflection in the sea! O throat! O throbbing heart! And I singing uselessly,
uselessly all the night. O past! O happy life! O songs of joy! In the air, in
the woods, over fields, Loved! loved! loved! loved! loved! But my mate no more,
no more with me! We two together no more. The aria sinking, All else continuing,
the stars shining, The winds blowing

In [59]:
poetry_query = "Which lines connect love, democracy, and religion?"

print_top_results_and_scores(
    query=poetry_query,
    embeddings=poetry_dataset["embeddings"],
    pages_and_chunks=poetry_dataset["chunks"],
    n_resources_to_return=3
)

Query: Which lines connect love, democracy, and religion?

Top retrieved chunks:

Score: 0.5837
Text:
Dear son do you think it is love? Listen dear son—listen America, daughter or
son, It is a painful thing to love a man or woman to excess, and yet it
satisfies, it is great, But there is something else very great, it makes the
whole coincide, It, magnificent, beyond materials, with continuous hands sweeps
and provides for all. Know you, solely to drop in the earth the germs of a
greater religion, The following chants each for its kind I sing. My comrade! For
you to share with me two greatnesses, and a third one rising inclusive and more
resplendent, The greatness of Love and Democracy, and the greatness of Religion.
Page number: 35
--------------------------------------------------------------------------------
Score: 0.5371
Text:
Each is not for its own sake, I say the whole earth and all the stars in the sky
are for religion’s sake. I say no man has ever yet been half devout enough, 

In [60]:
def prompt_formatter_poetry(query, context_items):
    """
    Formats retrieved poetry excerpts into a literary RAG prompt.
    """

    context = "\n\n".join(
        [
            f"Excerpt {i+1} | Page {item['page_number']} | Score {item['score']:.4f}\n"
            f"{item['sentence_chunk']}"
            for i, item in enumerate(context_items)
        ]
    )

    prompt = f"""
You are a careful literary assistant answering questions based on retrieved excerpts from a poetry PDF.

Use only the excerpts below.
If you interpret a theme, emotion, or symbol, clearly say that it is an interpretation based on the retrieved lines.
Do not claim the poet's exact intention unless it is directly stated.
Do not use outside knowledge about the poet or the book.
If the excerpts are not enough to answer, say:
"The document does not provide enough information."

Retrieved poetry excerpts:
{context}

Question:
{query}

Answer:
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

In [61]:
def ask_poetry(query, dataset, n_resources_to_return=3, max_new_tokens=256):
    """
    Full RAG pipeline for poetry:
    1. Retrieve relevant poetry excerpts
    2. Add retrieved excerpts to a literary prompt
    3. Generate an answer using the LLM
    """

    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=dataset["embeddings"],
        n_resources_to_return=n_resources_to_return
    )

    context_items = []

    for score, idx in zip(scores, indices):
        idx = idx.item()
        item = dataset["chunks"][idx].copy()
        item["score"] = score.item()
        context_items.append(item)

    prompt = prompt_formatter_poetry(
        query=query,
        context_items=context_items
    )

    model_device = next(llm_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer, context_items

In [62]:
query = "Which lines mention death or mortality?"

answer, context_items = ask_poetry(
    query=query,
    dataset=poetry_dataset,
    n_resources_to_return=3,
    max_new_tokens=256
)

print("Query:")
print(query)

print("\nPoetry RAG Answer:")
print(answer)

print("\nRetrieved Poetry Excerpts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print(item["sentence_chunk"])
    print("-" * 100)

Query:
Which lines mention death or mortality?

Poetry RAG Answer:
Based on the provided excerpts:

1. Excerpt 1 mentions "the dead" multiple times: "the streets, the shipping, the places of amusement, the Chicago, Boston, Philadelphia, the Mannahatta, were as full of the dead as of the living, And fuller, O vastly fuller of the dead than of the living;"
2. Excerpt 2 mentions "death" twice: "And I knew death, its thought, and the sacred knowledge of death."
3. Excerpt 3 mentions "death" once: "through me shall the words be said to make death exhilarating."

Retrieved Poetry Excerpts:
Score: 0.6522
Page number: 337
The houses full of life were equally full of death, (this house is now,)
The streets, the shipping, the places of amusement, the Chicago,
Boston, Philadelphia, the Mannahatta, were as full of the dead as of the living,
And fuller, O vastly fuller of the dead than of the living;
And what I dream’d I will henceforth tell to every person and age,
And I stand henceforth bound to 

In [63]:
query = "Which lines express loneliness or sadness?"

answer, context_items = ask_poetry(
    query=query,
    dataset=poetry_dataset,
    n_resources_to_return=3,
    max_new_tokens=256
)

print("Query:")
print(query)

print("\nPoetry RAG Answer:")
print(answer)

print("\nRetrieved Poetry Excerpts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print(item["sentence_chunk"])
    print("-" * 100)

Query:
Which lines express loneliness or sadness?

Poetry RAG Answer:
Based on the retrieved excerpts:

- Excerpt 1 mentions "lonesome walks" where someone thinks of their dear friends and loves, indicating loneliness.
- Excerpt 2 includes phrases like "pensive away," "sleepless and dissatisfied," and "no more with me," which convey feelings of sadness and loneliness.
- Excerpt 3 talks about "dulness of the years," "querilities," "ungracious glooms," "aches," "lethargy," "whimpering ennui," and "may filter in my dally songs," suggesting ongoing sadness and loneliness throughout life.

Retrieved Poetry Excerpts:
Score: 0.5901
Page number: 94
Who was not proud of his songs, but of the measureless ocean of love within him, and freely pour’d it
forth,
Who often walk’d lonesome walks thinking of his dear friends, his lovers,
Who pensive away from one he lov’d often lay sleepless and dissatisfied at night,
Who knew too well the sick, sick dread lest the one he lov’d might secretly be indiffe

In [64]:
def ask_from_dataset(query, dataset, n_resources_to_return=2, max_new_tokens=256):
    """
    Generic factual RAG pipeline for a selected dataset.
    This will be used for the Human Nutrition PDF.
    """

    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=dataset["embeddings"],
        n_resources_to_return=n_resources_to_return
    )

    context_items = []

    for score, idx in zip(scores, indices):
        idx = idx.item()
        item = dataset["chunks"][idx].copy()
        item["score"] = score.item()
        context_items.append(item)

    prompt = prompt_formatter(
        query=query,
        context_items=context_items
    )

    model_device = next(llm_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer, context_items

In [65]:
import pandas as pd

comparison_rows = []

# Factual textbook queries
nutrition_tests = [
    "What are the symptoms of pellagra?",
    "What are macronutrients, and what roles do they play in the human body?",
    "What is the difference between macronutrients and micronutrients?"
]

for query in nutrition_tests:
    answer, context_items = ask_from_dataset(
        query=query,
        dataset=nutrition_dataset,
        n_resources_to_return=2,
        max_new_tokens=256
    )

    comparison_rows.append({
        "Dataset": nutrition_dataset["name"],
        "Document Type": "Factual textbook",
        "Chunking Strategy": "Sentence-based chunking",
        "Query Type": "Factual question",
        "Query": query,
        "Chunk Count": len(nutrition_dataset["chunks"]),
        "Embedding Shape": str(tuple(nutrition_dataset["embeddings"].shape)),
        "Top Pages": [item["page_number"] for item in context_items],
        "Top Scores": [round(item["score"], 4) for item in context_items],
        "Answer Style": "Direct factual answer",
        "RAG Answer": answer
    })


# Literary poetry queries
poetry_tests = [
    "Which lines mention death or mortality?",
    "Which lines express loneliness or sadness?",
    "Which lines connect love, democracy, and religion?"
]

for query in poetry_tests:
    answer, context_items = ask_poetry(
        query=query,
        dataset=poetry_dataset,
        n_resources_to_return=3,
        max_new_tokens=256
    )

    comparison_rows.append({
        "Dataset": poetry_dataset["name"],
        "Document Type": "Literary poetry",
        "Chunking Strategy": "Line-based chunking",
        "Query Type": "Thematic / interpretive question",
        "Query": query,
        "Chunk Count": len(poetry_dataset["chunks"]),
        "Embedding Shape": str(tuple(poetry_dataset["embeddings"].shape)),
        "Top Pages": [item["page_number"] for item in context_items],
        "Top Scores": [round(item["score"], 4) for item in context_items],
        "Answer Style": "Interpretive answer based on retrieved excerpts",
        "RAG Answer": answer
    })

comparison_df = pd.DataFrame(comparison_rows)

pd.set_option("display.max_colwidth", 180)
comparison_df

,Dataset,Document Type,Chunking Strategy,Query Type,Query,Chunk Count,Embedding Shape,Top Pages,Top Scores,Answer Style,RAG Answer
0,Human Nutrition PDF,Factual textbook,Sentence-based chunking,Factual question,What are the symptoms of pellagra?,1680,"(1680, 768)","[565, 591]","[0.4655, 0.3476]",Direct factual answer,"The symptoms of pellagra include fatigue, decreased appetite, indigestion, diarrhea, dermatitis (skin issues), dementia, and sometimes death."
1,Human Nutrition PDF,Factual textbook,Sentence-based chunking,Factual question,"What are macronutrients, and what roles do they play in the human body?",1680,"(1680, 768)","[5, 8]","[0.7387, 0.7118]",Direct factual answer,"Macronutrients are nutrients that are needed in large amounts and include carbohydrates, lipids, and proteins. They can be metabolically processed into cellular energy and come..."
2,Human Nutrition PDF,Factual textbook,Sentence-based chunking,Factual question,What is the difference between macronutrients and micronutrients?,1680,"(1680, 768)","[5, 8]","[0.6544, 0.5499]",Direct factual answer,The main difference between macronutrients and micronutrients lies in the amount of these nutrients that the human body requires:\n\n- **Macronutrients** are nutrients that are...
3,Leaves of Grass Poetry PDF,Literary poetry,Line-based chunking,Thematic / interpretive question,Which lines mention death or mortality?,1352,"(1352, 768)","[337, 268, 90]","[0.6522, 0.6432, 0.6413]",Interpretive answer based on retrieved excerpts,"Based on the provided excerpts:\n\n1. Excerpt 1 mentions ""the dead"" multiple times: ""the streets, the shipping, the places of amusement, the Chicago, Boston, Philadelphia, the ..."
4,Leaves of Grass Poetry PDF,Literary poetry,Line-based chunking,Thematic / interpretive question,Which lines express loneliness or sadness?,1352,"(1352, 768)","[94, 186, 394]","[0.5901, 0.5706, 0.5683]",Interpretive answer based on retrieved excerpts,"Based on the retrieved excerpts:\n\n- Excerpt 1 mentions ""lonesome walks"" where someone thinks of their dear friends and loves, indicating loneliness.\n- Excerpt 2 includes phr..."
5,Leaves of Grass Poetry PDF,Literary poetry,Line-based chunking,Thematic / interpretive question,"Which lines connect love, democracy, and religion?",1352,"(1352, 768)","[35, 34, 92]","[0.5837, 0.5371, 0.4894]",Interpretive answer based on retrieved excerpts,"Based on the retrieved excerpts:\n\n- Excerpt 1: ""I sing. My comrade! For you to share with me two greatnesses, and a third one rising inclusive and more resplendent, The great..."


In [66]:
comparison_df.to_csv("nutrition_vs_poetry_rag_comparison.csv", index=False)
print("Comparison table saved.")

Comparison table saved.


In [67]:
pd.set_option("display.max_colwidth", None)
comparison_df

,Dataset,Document Type,Chunking Strategy,Query Type,Query,Chunk Count,Embedding Shape,Top Pages,Top Scores,Answer Style,RAG Answer
0,Human Nutrition PDF,Factual textbook,Sentence-based chunking,Factual question,What are the symptoms of pellagra?,1680,"(1680, 768)","[565, 591]","[0.4655, 0.3476]",Direct factual answer,"The symptoms of pellagra include fatigue, decreased appetite, indigestion, diarrhea, dermatitis (skin issues), dementia, and sometimes death."
1,Human Nutrition PDF,Factual textbook,Sentence-based chunking,Factual question,"What are macronutrients, and what roles do they play in the human body?",1680,"(1680, 768)","[5, 8]","[0.7387, 0.7118]",Direct factual answer,"Macronutrients are nutrients that are needed in large amounts and include carbohydrates, lipids, and proteins. They can be metabolically processed into cellular energy and come from the chemical bonds within these macronutrients. The energy from macronutrients is converted into cellular energy that is used to perform various bodily functions. Macronutrients are crucial for providing energy, aiding in metabolism, and supporting overall health."
2,Human Nutrition PDF,Factual textbook,Sentence-based chunking,Factual question,What is the difference between macronutrients and micronutrients?,1680,"(1680, 768)","[5, 8]","[0.6544, 0.5499]",Direct factual answer,"The main difference between macronutrients and micronutrients lies in the amount of these nutrients that the human body requires:\n\n- **Macronutrients** are nutrients that are needed in larger amounts. They include carbohydrates, lipids, and proteins. These nutrients are crucial because they help generate energy through metabolic processes and support various bodily functions.\n\n- **Micronutrients**, on the other hand, are nutrients that are needed in smaller amounts. Unlike macronutrients, which provide energy directly, micronutrients do not yield calories. Instead, they play vital roles in supporting bodily functions without providing direct energy. Examples of micronutrients include minerals like calcium, iron, zinc, and vitamins such as vitamin C, vitamin D, and B-complex vitamins.\n\nIn summary, while both types of nutrients are essential for health, macronutrients primarily serve as energy sources, whereas micronutrients contribute indirectly to overall function through their role as cofactors in enzymatic reactions."
3,Leaves of Grass Poetry PDF,Literary poetry,Line-based chunking,Thematic / interpretive question,Which lines mention death or mortality?,1352,"(1352, 768)","[337, 268, 90]","[0.6522, 0.6432, 0.6413]",Interpretive answer based on retrieved excerpts,"Based on the provided excerpts:\n\n1. Excerpt 1 mentions ""the dead"" multiple times: ""the streets, the shipping, the places of amusement, the Chicago, Boston, Philadelphia, the Mannahatta, were as full of the dead as of the living, And fuller, O vastly fuller of the dead than of the living;""\n2. Excerpt 2 mentions ""death"" twice: ""And I knew death, its thought, and the sacred knowledge of death.""\n3. Excerpt 3 mentions ""death"" once: ""through me shall the words be said to make death exhilarating."""
4,Leaves of Grass Poetry PDF,Literary poetry,Line-based chunking,Thematic / interpretive question,Which lines express loneliness or sadness?,1352,"(1352, 768)","[94, 186, 394]","[0.5901, 0.5706, 0.5683]",Interpretive answer based on retrieved excerpts,"Based on the retrieved excerpts:\n\n- Excerpt 1 mentions ""lonesome walks"" where someone thinks of their dear friends and loves, indicating loneliness.\n- Excerpt 2 includes phrases like ""pensive away,"" ""sleepless and dissatisfied,"" and ""no more with me,"" which convey feelings of sadness and loneliness.\n- Excerpt 3 talks about ""dulness of the years,"" ""querilities,"" ""ungracious glooms,"" ""aches,"" ""lethargy,"" ""whimpering ennui,"" and ""may filter in my dally songs,"" suggesting ongoing sadness and loneliness throughout life."
5,Leaves of Grass Poetry PDF,Literary poetr

In [68]:
compact_comparison_df = comparison_df[
    [
        "Dataset",
        "Document Type",
        "Chunking Strategy",
        "Query Type",
        "Query",
        "Chunk Count",
        "Top Scores",
        "Answer Style"
    ]
]

compact_comparison_df

,Dataset,Document Type,Chunking Strategy,Query Type,Query,Chunk Count,Top Scores,Answer Style
0,Human Nutrition PDF,Factual textbook,Sentence-based chunking,Factual question,What are the symptoms of pellagra?,1680,"[0.4655, 0.3476]",Direct factual answer
1,Human Nutrition PDF,Factual textbook,Sentence-based chunking,Factual question,"What are macronutrients, and what roles do they play in the human body?",1680,"[0.7387, 0.7118]",Direct factual answer
2,Human Nutrition PDF,Factual textbook,Sentence-based chunking,Factual question,What is the difference between macronutrients and micronutrients?,1680,"[0.6544, 0.5499]",Direct factual answer
3,Leaves of Grass Poetry PDF,Literary poetry,Line-based chunking,Thematic / interpretive question,Which lines mention death or mortality?,1352,"[0.6522, 0.6432, 0.6413]",Interpretive answer based on retrieved excerpts
4,Leaves of Grass Poetry PDF,Literary poetry,Line-based chunking,Thematic / interpretive question,Which lines express loneliness or sadness?,1352,"[0.5901, 0.5706, 0.5683]",Interpretive answer based on retrieved excerpts
5,Leaves of Grass Poetry PDF,Literary poetry,Line-based chunking,Thematic / interpretive question,"Which lines connect love, democracy, and religion?",1352,"[0.5837, 0.5371, 0.4894]",Interpretive answer based on retrieved excerpts


In [69]:
import os
import requests

second_pdf_path = "attention_is_all_you_need.pdf"

url = "https://arxiv.org/pdf/1706.03762.pdf"

if not os.path.exists(second_pdf_path):
    response = requests.get(url)
    with open(second_pdf_path, "wb") as f:
        f.write(response.content)
    print("Second PDF downloaded successfully.")
else:
    print("Second PDF already exists.")

Second PDF downloaded successfully.


In [70]:
import fitz
import re
import pandas as pd
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    return text.replace("\n", " ").replace("\xa0", " ").strip()


def open_and_read_pdf_v2(pdf_path: str, page_offset: int = 0) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages_and_texts = []

    for page_number, page in tqdm(enumerate(doc), total=len(doc)):
        text = page.get_text()
        text = text_formatter(text)

        pages_and_texts.append({
            "page_number": page_number + page_offset,
            "page_char_count": len(text),
            "page_word_count": len(text.split(" ")),
            "page_sentence_count_raw": len(text.split(". ")),
            "page_token_count": len(text) / 4,
            "text": text
        })

    doc.close()
    return pages_and_texts


def split_list(input_list: list, slice_size: int) -> list[list[str]]:
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]


def build_rag_dataset_from_pdf(
    pdf_path: str,
    dataset_name: str,
    embedding_model,
    device,
    page_offset: int = 0,
    num_sentence_chunk_size: int = 10,
    min_token_length: int = 30
):
    """
    Turns a PDF into chunks and embeddings for RAG.
    """

    # 1. Read PDF
    pages_and_texts = open_and_read_pdf_v2(
        pdf_path=pdf_path,
        page_offset=page_offset
    )

    # 2. Sentence splitting
    for item in tqdm(pages_and_texts):
        item["sentences"] = list(nlp(item["text"]).sents)
        item["sentences"] = [str(sentence) for sentence in item["sentences"]]
        item["page_sentence_count_spacy"] = len(item["sentences"])

    # 3. Chunking
    for item in tqdm(pages_and_texts):
        item["sentence_chunks"] = split_list(
            input_list=item["sentences"],
            slice_size=num_sentence_chunk_size
        )
        item["num_chunks"] = len(item["sentence_chunks"])

    # 4. Convert chunks to list of dictionaries
    pages_and_chunks_new = []

    for item in tqdm(pages_and_texts):
        for sentence_chunk in item["sentence_chunks"]:
            chunk_dict = {}

            joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
            joined_sentence_chunk = re.sub(r"\.([A-Z])", r". \1", joined_sentence_chunk)

            chunk_dict["page_number"] = item["page_number"]
            chunk_dict["sentence_chunk"] = joined_sentence_chunk
            chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
            chunk_dict["chunk_word_count"] = len(joined_sentence_chunk.split(" "))
            chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4

            pages_and_chunks_new.append(chunk_dict)

    # 5. Filter short chunks
    df_chunks = pd.DataFrame(pages_and_chunks_new)

    pages_and_chunks_over_min_token_len = df_chunks[
        df_chunks["chunk_token_count"] > min_token_length
    ].to_dict(orient="records")

    # 6. Create embeddings
    text_chunks = [
        item["sentence_chunk"]
        for item in pages_and_chunks_over_min_token_len
    ]

    chunk_embeddings = embedding_model.encode(
        text_chunks,
        batch_size=32,
        convert_to_tensor=True
    ).to(device)

    dataset = {
        "name": dataset_name,
        "pdf_path": pdf_path,
        "chunks": pages_and_chunks_over_min_token_len,
        "embeddings": chunk_embeddings
    }

    print(f"Dataset name: {dataset_name}")
    print(f"Number of chunks: {len(dataset['chunks'])}")
    print(f"Embedding shape: {dataset['embeddings'].shape}")

    return dataset

In [71]:
attention_dataset = build_rag_dataset_from_pdf(
    pdf_path=second_pdf_path,
    dataset_name="Attention Is All You Need PDF",
    embedding_model=embedding_model,
    device=device,
    page_offset=0,
    num_sentence_chunk_size=10,
    min_token_length=30
)

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

Dataset name: Attention Is All You Need PDF
Number of chunks: 40
Embedding shape: torch.Size([40, 768])


In [72]:
def ask_from_dataset(query, dataset, n_resources_to_return=2, max_new_tokens=256):
    """
    Full RAG pipeline for a selected dataset.
    """

    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=dataset["embeddings"],
        n_resources_to_return=n_resources_to_return
    )

    context_items = []

    for score, idx in zip(scores, indices):
        idx = idx.item()
        item = dataset["chunks"][idx].copy()
        item["score"] = score.item()
        context_items.append(item)

    prompt = prompt_formatter(
        query=query,
        context_items=context_items
    )

    model_device = next(llm_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer, context_items

compare 2 datasets

In [73]:
comparison_tests = [
    {
        "dataset": nutrition_dataset,
        "query": "What are the symptoms of pellagra?"
    },
    {
        "dataset": nutrition_dataset,
        "query": "What are macronutrients, and what roles do they play in the human body?"
    },
    {
        "dataset": attention_dataset,
        "query": "What is self-attention?"
    },
    {
        "dataset": attention_dataset,
        "query": "What is the Transformer architecture?"
    }
]

comparison_results = []

for test in comparison_tests:
    dataset = test["dataset"]
    query = test["query"]

    answer, context_items = ask_from_dataset(
        query=query,
        dataset=dataset,
        n_resources_to_return=2,
        max_new_tokens=256
    )

    comparison_results.append({
        "Dataset": dataset["name"],
        "Query": query,
        "Number of Chunks": len(dataset["chunks"]),
        "Embedding Shape": str(tuple(dataset["embeddings"].shape)),
        "Top Retrieved Pages": [item["page_number"] for item in context_items],
        "Top Scores": [round(item["score"], 4) for item in context_items],
        "RAG Answer": answer
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df

,Dataset,Query,Number of Chunks,Embedding Shape,Top Retrieved Pages,Top Scores,RAG Answer
0,Human Nutrition PDF,What are the symptoms of pellagra?,1680,"(1680, 768)","[565, 591]","[0.4655, 0.3476]","The symptoms of pellagra include fatigue, decreased appetite, indigestion, diarrhea, dermatitis (skin issues), dementia, and sometimes death."
1,Human Nutrition PDF,"What are macronutrients, and what roles do they play in the human body?",1680,"(1680, 768)","[5, 8]","[0.7387, 0.7118]","Macronutrients are nutrients that are needed in large amounts and include carbohydrates, lipids, and proteins. They can be metabolically processed into cellular energy and come from the chemical bonds within these macronutrients. The energy from macronutrients is converted into cellular energy that is used to perform various bodily functions. Macronutrients are crucial for providing energy, aiding in metabolism, and supporting overall health."
2,Attention Is All You Need PDF,What is self-attention?,40,"(40, 768)","[12, 0]","[0.3636, 0.312]",The passage does not define what self-attention is.
3,Attention Is All You Need PDF,What is the Transformer architecture?,40,"(40, 768)","[2, 4]","[0.5269, 0.3904]","The Transformer architecture consists of an encoder and a decoder, both composed of stacks of identical layers. Each layer includes two sub-layers: a multi-head self-attention mechanism and a simple, position-wise fully connected feed-forward network. The model employs residual connections and layer normalization around each sub-layer. All sub-layers in the model, including the embedding layers, have outputs of dimension \(d_{\text{model}} = 512\)."


cross-dataset sanity check

In [74]:
query = "What are the symptoms of pellagra?"

answer, context_items = ask_from_dataset(
    query=query,
    dataset=attention_dataset,
    n_resources_to_return=2,
    max_new_tokens=256
)

print("Dataset:", attention_dataset["name"])
print("Query:", query)
print("\nAnswer:")
print(answer)

print("\nRetrieved Contexts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print_wrapped(item["sentence_chunk"])
    print("-" * 80)

Dataset: Attention Is All You Need PDF
Query: What are the symptoms of pellagra?

Answer:
The symptoms of pellagra include dermatitis, dementia, diarrhea, and dementia.

Retrieved Contexts:
Score: 0.0189
Page number: 13
The Law will never be perfect , but its application should be just - this is
what we are missing , in my opinion .<EOS> <pad> The Law will never be perfect ,
but its application should be just - this is what we are missing , in my opinion
.<EOS> <pad> The Law will never be perfect , but its application should be just
- this is what we are missing , in my opinion .<EOS> <pad> The Law will never be
perfect , but its application should be just - this is what we are missing , in
my opinion .<EOS> <pad> Figure 4: Two attention heads, also in layer 5 of 6,
apparently involved in anaphora resolution. Top: Full attentions for head 5.
Bottom: Isolated attentions from just the word ‘its’ for attention heads 5 and
6. Note that the attentions are very sharp for this word.14
-------